# Scamurai — multilingual lexical / semantic evaluation

**Experimental diagnostic only.** Use synthetic or authorized non-sensitive labelled pairs. This notebook is not production ingestion, permanent case storage, an API service, always-on alerting, or a coordination/truth classifier. Do not enter PATs, provider credentials, private model tokens, or sensitive case text.

The bundled fixture is deliberately small and synthetic. Its scores are not production accuracy or calibration. Official references: [Sentence Transformers pairwise similarity](https://www.sbert.net/docs/sentence_transformer/usage/semantic_textual_similarity.html) and [Colab FAQ/resource limits](https://research.google.com/colaboratory/faq.html).

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO = Path('/content/scamurai')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/sudonishant/scamurai.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'backend/requirements.txt')], check=True)
FIXTURE = REPO / 'backend/fixtures/multilingual_narratives.json'
fixture = json.loads(FIXTURE.read_text(encoding='utf-8'))
assert fixture['synthetic'] is True
print(f"Loaded {len(fixture['pairs'])} explicitly synthetic diagnostic pairs.")

In [ ]:
# Run the repository's actual lexical diagnostic; no semantic model is loaded.
lexical_output = Path('/tmp/scamurai-lexical-evaluation.json')
subprocess.run([
    sys.executable, str(REPO / 'backend/scripts/evaluate_multilingual.py'),
    '--output', str(lexical_output),
], cwd=REPO / 'backend', check=True)
lexical_report = json.loads(lexical_output.read_text(encoding='utf-8'))
assert lexical_report['method']['semantic_embedding_model'] == 'not_configured'

## Optional reviewed multilingual model

The next cells do nothing until you provide an exact **public** model ID. Review the model card, license, supported languages, training/evaluation data and intended similarity function first. Prefer a pinned revision for reproducibility. Do not use a private model token in this notebook.

Leave `EVALUATION_THRESHOLD = None` to inspect scores without classification claims. If you set a threshold, derive it from representative held-out validation data—not from Scamurai's lexical `0.48` threshold—and report false positives/negatives by language and control type.

In [ ]:
MODEL_ID = ''  # Example format: 'organization/exact-model-name'; no model is selected by default.
MODEL_REVISION = None  # Prefer an exact reviewed commit/revision for reproducibility.
EVALUATION_THRESHOLD = None  # Numeric only after separate validation; never inherit lexical 0.48.

In [ ]:
if MODEL_ID:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True)
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION)
    left = model.encode([row['left'] for row in fixture['pairs']])
    right = model.encode([row['right'] for row in fixture['pairs']])
    scores = model.similarity_pairwise(left, right).detach().cpu().tolist()
    rows = []
    for pair, score in zip(fixture['pairs'], scores):
        row = {
            'id': pair['id'], 'track': pair['track'], 'languages': pair['languages'],
            'label': pair['label'], 'model_similarity': float(score),
        }
        if EVALUATION_THRESHOLD is not None:
            row['predicted_same_candidate'] = bool(score >= EVALUATION_THRESHOLD)
            row['correct'] = row['predicted_same_candidate'] == (pair['label'] == 'same_narrative')
        rows.append(row)

    report = {
        'synthetic_diagnostic': True,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'similarity_function': str(model.similarity_fn_name),
        'threshold': EVALUATION_THRESHOLD,
        'threshold_status': 'score_only' if EVALUATION_THRESHOLD is None else 'experimental_requires_holdout_validation',
        'pairs': rows,
        'limitations': [
            'Small synthetic diagnostic, not production accuracy or calibration.',
            'Similarity is not truth, coordination, malicious intent, or common ownership.',
            'Validate on representative event/account-separated data before any operational use.',
        ],
    }
    output = Path('/tmp/scamurai-semantic-evaluation.json')
    output.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps({key: value for key, value in report.items() if key != 'pairs'}, indent=2))
    print(f'Wrote ephemeral pair scores to {output}; inspect false candidates before drawing conclusions.')
else:
    print('Semantic experiment not run: set a reviewed public MODEL_ID first. No semantic result is claimed.')

## Required next evaluation step

Replace or supplement the synthetic fixture only with authorized, non-sensitive, representative labelled pairs. Split by event and account to prevent near-duplicate leakage. Include translations, transliteration, code-switching, quotations, breaking-news/press-release controls, different-place/status templates and access-coverage notes. Report per-group precision/recall, false positives/negatives, calibration where applicable, model/revision and threshold provenance. Do not deploy a threshold solely because it improves this notebook's fixture.